# Practical 11: Bayesian & Frequentist Gamma Modeling of Sunspot Counts

- **Input Data:** `../Datasets/sunspots.csv` (Monthly mean total sunspot observations, Jan 1749 – Nov 2018)
- **Goals:** Visualize long-term solar cycles, analyze 12-year folded cycle profiles, estimate Gamma shape ($\alpha$) and scale ($\beta$) parameters via Maximum Likelihood and log-space Metropolis-Hastings MCMC, and generate posterior predictive intervals.
- **Zero-Count Adjustment:** Because the Gamma density is strictly positive ($y > 0$), all monthly counts are shifted by $+1.0$ prior to likelihood evaluation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, special

plt.style.use('seaborn-v0_8-whitegrid')

csv_path = '../Datasets/sunspots.csv' if os.path.exists('../Datasets/sunspots.csv') else 'Datasets/sunspots.csv'
raw_df = pd.read_csv(csv_path)
date_col = [col for col in raw_df.columns if 'date' in col.lower()][0]
val_col = [col for col in raw_df.columns if col != date_col][-1]

sun_df = pd.DataFrame({
    'date': pd.to_datetime(raw_df[date_col]),
    'sunspots': pd.to_numeric(raw_df[val_col], errors='coerce')
}).dropna().reset_index(drop=True)

counts = sun_df['sunspots'].to_numpy()
print(f'Total Observations : {len(sun_df)}')
print(f'Date Span          : {sun_df["date"].min():%B %Y} to {sun_df["date"].max():%B %Y}')
sun_df.head()

### Stage 1: Historical Time-Series Exploration

In [ ]:
peak_idx = sun_df['sunspots'].idxmax()
peak_date = sun_df.loc[peak_idx, 'date'].strftime('%B %Y')
peak_val = sun_df.loc[peak_idx, 'sunspots']

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(sun_df['date'], counts, color='#d95f02', linewidth=0.75, label='Monthly Sunspot Number')
ax.axhline(counts.mean(), color='#1b9e77', linestyle='--', linewidth=1.2, label=f'Mean ({counts.mean():.2f})')
ax.set_title('Monthly Mean Total Sunspot Number (1749–2018)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Sunspot Number')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Sample Mean  : {counts.mean():.2f}')
print(f'Sample Peak  : {peak_val:.1f} ({peak_date})')

### Stage 2: 12-Year Epoch Folding & Gamma Curve Envelope

In [ ]:
sun_df['idx'] = np.arange(len(sun_df))
sun_df['block_12yr'] = sun_df['idx'] // 144
sun_df['year_offset'] = (sun_df['idx'] % 144 + 0.5) / 12.0

block_summary = sun_df.groupby('block_12yr')['sunspots'].agg(Mean='mean', Peak='max').round(2)
print('Total 12-year blocks:', len(block_summary))
print(block_summary.head(8))

full_blocks = sun_df[sun_df['block_12yr'] < sun_df['block_12yr'].max()]
mean_profile = full_blocks.groupby('year_offset')['sunspots'].mean()
t_axis = mean_profile.index.to_numpy()

def scaled_gamma_pdf(x, scale_factor, alpha, beta):
    return scale_factor * stats.gamma.pdf(x, alpha, scale=beta)

popt_env, _ = optimize.curve_fit(
    scaled_gamma_pdf,
    t_axis,
    mean_profile.to_numpy(),
    p0=[mean_profile.sum(), 2.5, 2.0],
    bounds=(1e-3, np.inf),
    maxfev=25000,
)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(t_axis, mean_profile.to_numpy(), color='#555555', linewidth=1.8, label='Mean 12-Year Profile')
ax.plot(t_axis, scaled_gamma_pdf(t_axis, *popt_env), color='#e7298a', linewidth=2.2, linestyle='--', label='Fitted Gamma Curve')
ax.set_title('Average 12-Year Block Profile with Gamma Curve Fit', fontweight='bold')
ax.set_xlabel('Years Since Block Start')
ax.set_ylabel('Mean Sunspot Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Fitted Envelope Shape (a): {popt_env[1]:.3f} | Scale (b): {popt_env[2]:.3f}')

### Stage 3: Shifted Gamma Maximum Likelihood Benchmark

In [ ]:
y_shifted = counts + 1.0
mle_a, _, mle_b = stats.gamma.fit(y_shifted, floc=0)
x_grid = np.linspace(y_shifted.min(), y_shifted.max(), 450)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.hist(y_shifted, bins=42, density=True, color='#7570b3', alpha=0.65, edgecolor='white', label='Observed (Counts + 1)')
ax.plot(x_grid, stats.gamma.pdf(x_grid, mle_a, scale=mle_b), color='#d95f02', linewidth=2.2, label=f'Gamma MLE (a={mle_a:.3f}, b={mle_b:.3f})')
ax.set_title('Empirical Sunspot Density vs. Maximum Likelihood Gamma Fit', fontweight='bold')
ax.set_xlabel('Shifted Sunspot Number (y + 1)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print(f'MLE Shape a : {mle_a:.4f} | MLE Scale b : {mle_b:.4f}')
print(f'Skewness    : {stats.skew(counts):.4f}')

### Stage 4: Log-Space Metropolis-Hastings Sampler Across Subsets

In [ ]:
def sample_gamma_posterior(y_obs, a_init=3.5, b_init=12.0, n_steps=7500, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_obs)
    sum_y = np.sum(y_obs)
    sum_log_y = np.sum(np.log(y_obs))
    step_sd = 1.8 / np.sqrt(n)

    def log_target(log_a, log_b):
        a, b = np.exp(log_a), np.exp(log_b)
        if a <= 1e-6 or b <= 1e-6 or a > 1e4 or b > 1e5:
            return -np.inf
        ll = (a - 1.0) * sum_log_y - (sum_y / b) - n * (a * log_b + special.gammaln(a))
        return ll + log_a + log_b

    cur_la, cur_lb = np.log(a_init), np.log(b_init)
    cur_lp = log_target(cur_la, cur_lb)
    chain = np.empty((n_steps, 2))
    accepted = 0

    for i in range(n_steps):
        prop_la = cur_la + rng.normal(0.0, step_sd)
        prop_lb = cur_lb + rng.normal(0.0, step_sd)
        prop_lp = log_target(prop_la, prop_lb)
        if np.log(rng.uniform()) < (prop_lp - cur_lp):
            cur_la, cur_lb, cur_lp = prop_la, prop_lb, prop_lp
            accepted += 1
        chain[i] = [np.exp(cur_la), np.exp(cur_lb)]

    return chain, accepted / n_steps

subsets = {
    'First 50 Months': y_shifted[:50],
    'All 3,239 Months': y_shifted,
    'Last 50 Months': y_shifted[-50:],
}

burn_in = 1500
chains_dict = {}
table_rows = []

for label, data_sub in subsets.items():
    ch, acc = sample_gamma_posterior(data_sub, seed=42)
    chains_dict[label] = ch
    post = ch[burn_in:]
    table_rows.append({
        'Subset': label,
        'n': len(data_sub),
        'Accept Rate': round(acc, 3),
        'Mean a': round(post[:, 0].mean(), 3),
        'SD(a)': round(post[:, 0].std(), 3),
        'Mean b': round(post[:, 1].mean(), 3),
        'SD(b)': round(post[:, 1].std(), 3),
    })

print(pd.DataFrame(table_rows).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#1b9e77', '#7570b3', '#e7298a']
for ax, (label, ch), col in zip(axes, chains_dict.items(), colors):
    post = ch[burn_in:]
    ax.scatter(post[:, 0], post[:, 1], s=4, alpha=0.25, color=col, label='Posterior Draws')
    ax.scatter(3.5, 12.0, color='black', marker='X', s=65, label='Start (3.5, 12.0)')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Shape (a)')
    ax.set_ylabel('Scale (b)')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Stage 5: Trace Diagnostics & Posterior Marginal Distributions

In [ ]:
full_chain = chains_dict['All 3,239 Months']
a_post, b_post = full_chain[burn_in:, 0], full_chain[burn_in:, 1]

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
axes[0, 0].plot(full_chain[:, 0], color='#1b9e77', linewidth=0.6)
axes[0, 0].axvline(burn_in, color='crimson', linestyle='--', label='Burn-in End')
axes[0, 0].set_title('Trace of Shape Parameter (a)')
axes[0, 0].set_ylabel('a')
axes[0, 0].legend()

axes[1, 0].plot(full_chain[:, 1], color='#7570b3', linewidth=0.6)
axes[1, 0].axvline(burn_in, color='crimson', linestyle='--', label='Burn-in End')
axes[1, 0].set_title('Trace of Scale Parameter (b)')
axes[1, 0].set_xlabel('Step')
axes[1, 0].set_ylabel('b')
axes[1, 0].legend()

axes[0, 1].hist(a_post, bins=35, color='#1b9e77', alpha=0.8, edgecolor='white')
axes[0, 1].axvline(mle_a, color='black', linestyle=':', linewidth=1.5, label=f'MLE ({mle_a:.3f})')
axes[0, 1].set_title('Posterior Histogram of a')
axes[0, 1].set_xlabel('a')
axes[0, 1].legend()

axes[1, 1].hist(b_post, bins=35, color='#7570b3', alpha=0.8, edgecolor='white')
axes[1, 1].axvline(mle_b, color='black', linestyle=':', linewidth=1.5, label=f'MLE ({mle_b:.3f})')
axes[1, 1].set_title('Posterior Histogram of b')
axes[1, 1].set_xlabel('b')
axes[1, 1].legend()
plt.tight_layout()
plt.show()

### Stage 6: Posterior Credible Intervals & Predictive Simulation

In [ ]:
a_hat, b_hat = a_post.mean(), b_post.mean()
a_ci = np.percentile(a_post, [2.5, 97.5])
b_ci = np.percentile(b_post, [2.5, 97.5])

print(f'Posterior a : {a_hat:.4f} (95% CI: {a_ci[0]:.4f} to {a_ci[1]:.4f})')
print(f'Posterior b : {b_hat:.4f} (95% CI: {b_ci[0]:.4f} to {b_ci[1]:.4f})')

rng = np.random.default_rng(99)
idx_draws = rng.integers(0, len(a_post), size=6000)
sim_future = rng.gamma(a_post[idx_draws], b_post[idx_draws]) - 1.0
pred_mean = a_hat * b_hat - 1.0
pred_range = np.percentile(sim_future, [2.5, 97.5])

print(f'Expected Monthly Sunspots : {pred_mean:.2f} (Observed Mean: {counts.mean():.2f})')
print(f'95% Predictive Range      : {pred_range[0]:.2f} to {pred_range[1]:.2f}')

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.hist(counts, bins=42, density=True, alpha=0.6, color='#66a61e', edgecolor='white', label='Observed')
ax.hist(sim_future, bins=42, density=True, alpha=0.5, color='#e6ab02', edgecolor='white', label='Posterior Predictive')
ax.set_title('Observed vs. Posterior Predictive Sunspot Distribution', fontweight='bold')
ax.set_xlabel('Sunspot Number')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()